<a href="https://colab.research.google.com/github/FOUEGAP/Rendement-la-maturit-/blob/main/Modelisation_Fouegap_Memoire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import datetime
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IVGMM
from linearmodels import PanelOLS
from linearmodels import IV2SLS
from collections import OrderedDict
from linearmodels.panel import compare
import pandas_datareader.data as web
from scipy.stats.mstats import winsorize

## I. DES DONNEES

On telecharge les données des actions des pays developpés et des pays émergents dans LSEG Workspace. Cette base est choisi parcequ'elle propose un accès aux données des deux catégories de pays. Les variables téléchargés sont :

Données du marchés :

`prc_close` : prix de clôture <br>
`prc_open` : prix bid d'ouverture<br>
`bid` : prix bid<br>
`ask` : prix ask<br>
`prc_high` : prix le plus haut<br>
`prc_low` : prix le plus bas<br>
`pe_ratio` : ratio p/e<br>
`price_to_book` : ratio prix du marché sur prix comptable<br>
`tot_asset` : total des actifs<br>
`tot_liab` : total dette<br>
`net_income` : revenu net<br>
`capex` : depenses en capital<br>
`tot_cur_asset` : total des actifs actruels <br>
`tot_cur_liab` : total dette actuelle<br>

In [ ]:
# pays developpes
dfd1 = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2005_2015/dfd1.csv') # shares of advanced countries 2005-2015
dfd = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2015_2025/dfd.csv') # shares of advanced countries  2015-2025
df_outd = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2005_2015/df_de_.csv') # number of shares outstanding of advanced countries 2005-2025

# pays emergents
dfe1 = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2005_2015/dfe1.csv') # shares of emerging countries 2005-2015
dfe = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2015_2025/dfe.csv') # shares of emerging countries 2015-2025
df_oute1 = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2005_2015/df_em_.csv') # number of shares outstanding of emerging countries 2005-2015
df_oute2 = pd.read_csv('C:/Users/Subtech/Desktop/data_final_memoire/data_2005_2015/df_em1111_.csv') # number of shares outstanding of emerging countries 2015-2025




df_oute1.rename(columns={
    'Date': 'date',
    'Outstanding Shares': 'RIC',
    '0': 'share_out'
}, inplace=True)

df_oute2.rename(columns={
    'Date': 'date',
    'Outstanding Shares': 'RIC',
    '0': 'share_out'
}, inplace=True)


df_outd.rename(columns={
    'Date': 'date',
    'Outstanding Shares': 'share_out',
}, inplace=True)
df_outd.drop(columns=['Institutional Ownership %'], inplace=True)

# concatener les deux dataframes
df_deve = pd.concat([dfd1, dfd], ignore_index=True) # concat df of developed countries 2005-2015 and 2015-2025
df_emer = pd.concat([dfe1, dfe], ignore_index=True) # concat df of emerging countries 2005-2015 and 2015-2025

df_oute = pd.concat([df_oute1, df_oute2], ignore_index=True) # concat df of number of shares outstanding of emerging countries 2005-2015 and 2015-2025

df_deve.rename(columns={'Date_x': 'date'}, inplace=True)
df_deve_ = pd.merge(df_deve, df_outd, on=['date', 'RIC'], how='left')
# merge df_emer et df_oute
df_emer.rename(columns={'Date_x': 'date'}, inplace=True)
df_emer_ = pd.merge(df_emer, df_oute, on=['date', 'RIC'], how='left')

df_deve_emer_ = pd.concat([df_deve_, df_emer_], ignore_index=True) # concat df of developed and emerging countries


df = df_deve_emer_.copy()

In [ ]:
del df_deve_
del df_deve
del df_emer_
del df_emer
del df_oute1
del df_oute2
del df_deve_emer_
del dfe
del dfe1
del dfd1
del dfd
del df_outd
del df_oute


In [ ]:
df

In [ ]:
# remplacer les infinies par des NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
# compter les infinies dans dfd
df.isin([np.inf, -np.inf]).sum()

In [ ]:
# renommer toutes les colonnes du dataframe dfd et dfe
df = df.rename(columns={'Price Close': 'prc_close', 'Volume': 'volume', 'Turnover': 'echange_val', 'Open Price': 'prc_open', 'Bid Price': 'bid', 'Ask Price': 'ask', 'High Price': 'prc_high', 'Low Price': 'prc_low', 'P/E(Time Series Ratio)': 'pe_ratio', 'Price To Book Value Per Share(Time Series Ratio)': 'price_to_book', 'Total Assets': 'tot_asset', 'Total Liabilities': 'tot_liab', 'Net Income Incl Extra Before Distributions': 'net_income', 'Capital Expenditures - Total per Share': 'capex', 'Total Current Assets': 'tot_cur_asset', 'Total Current Liabilities': 'tot_cur_liab'})
# convertir la colonne Date en datetime
df['date'] = pd.to_datetime(df['date'])

# Calculer cetaines variables dfd
df['tot_asset'] = df['tot_asset'].replace(0, np.nan)
df['tot_cur_liab'] = df['tot_cur_liab'].replace(0, np.nan)
df['share_out'] = df['share_out'].replace(0, np.nan)
df['prc_close'] = df['prc_close'].replace(0, np.nan)
df['volume'] = df['volume'].replace(0, np.nan)


df['size'] = np.log(df['tot_asset'])
df['ROA'] = df['net_income'] / df['tot_asset']
df['leverage'] = df['tot_liab'] / df['tot_asset']
df['liquidity'] = df['tot_cur_asset'] / df['tot_cur_liab']
df['turnover'] = df['volume'] /  df['share_out']
df['market_cap'] = df['prc_close'] * df['share_out']
df['market_cap_log'] = np.log(df['market_cap'])


In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

# II. Mesure des couts de transaction

In [ ]:
# calcul des rendements
df = df.sort_values(by=['RIC', 'date'])

# Calcul du rendement log (ou rendement simple, à toi de choisir)
# Option 1 : rendement simple
df['ret_'] = df.groupby('RIC')['prc_close'].transform(lambda x: x.pct_change(fill_method=None))
# Option 1 : rendement log
df['ret'] = df.groupby('RIC')['prc_close'].transform(lambda x: np.log(x / x.shift(1)))

# calcul variable le spread en pourcentage avec le bid et ask
df['mid_bid_ask'] = (df['bid'] + df['ask']) / 2
df['mid_bid_ask'] = df['mid_bid_ask'].replace(0, np.nan)
df['Spread_bid_ask'] = (df['ask'] - df['bid']) / df['mid_bid_ask']
# calcul variable le spread en pourcentage avec le high et low
df['mid_hi_lo'] = (df['prc_high'] + df['prc_low']) / 2
df['mid_hi_lo'] = df['mid_hi_lo'].replace(0, np.nan)
df['Spread_hi_lo'] = (df['prc_high'] - df['prc_low']) / df['mid_hi_lo']

In [ ]:
#  Fonction personnalisée pour mesurer la covariance lag-1
def roll_measure_vec(x: np.ndarray) -> float:
    if len(x) < 2 or np.all(np.isnan(x)):
        return np.nan

    x1, x2 = x[1:], x[:-1]
    cov = np.cov(x1, x2, bias=True)[0, 1]

    return 2 * np.sqrt(abs(cov))

#  Configuration de la fenêtre
window_size = 10

#  Trier les données par RIC et date
df = df.sort_values(['RIC', 'date'])

# Calcul vectorisé de roll_measure par RIC
df['roll_measure'] = (
    df
    .groupby('RIC')['ret']
    .rolling(window_size)
    .apply(roll_measure_vec, raw=True)
    .reset_index(level=0, drop=True)  # Nettoie l'index pour coller au DataFrame original
)

In [ ]:
# Remplacer les zéros par NaN (sans drop)
df['prc_high'] = df['prc_high'].replace(0, np.nan)
df['prc_low'] = df['prc_low'].replace(0, np.nan)

# Trier les données
df = df.sort_values(by=['RIC', 'date']).reset_index(drop=True)

# Décalages par RIC
df['prc_high_lag1'] = df.groupby('RIC')['prc_high'].shift(1)
df['prc_low_lag1'] = df.groupby('RIC')['prc_low'].shift(1)

# Constante Corwin & Schultz
denom = 3 - 2 * np.sqrt(2)

# Calculs vectorisés
log_hl_t   = np.log(df['prc_high'] / df['prc_low'])
log_hl_t1  = np.log(df['prc_high_lag1'] / df['prc_low_lag1'])
beta       = log_hl_t**2 + log_hl_t1**2

high_max = np.maximum(df['prc_high'], df['prc_high_lag1'])
low_min  = np.minimum(df['prc_low'], df['prc_low_lag1'])
gamma    = np.log(high_max / low_min)**2

# Calcul alpha
alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / denom - np.sqrt(gamma / denom)

# Calcul du spread estimé S
S = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))

# Sécurité : Ne garder que les cas valides
S[~np.isfinite(S)] = np.nan  # remplace inf, -inf, nan par NaN

# Affectation finale
df['S'] = S


# Variables Macro et de marché

In [ ]:
start = datetime.datetime(2005, 1, 1)
end = datetime.datetime(2025, 1, 1)

# Télécharger les séries FRED
vix = web.DataReader('VIXCLS', 'fred', start, end) #vix sur spx

# Load 1-month T-bill rate (DGS1MO)
rf_fred = web.DataReader('DGS1MO', 'fred', start, end) #taux annuel des obligations 1mo (selon Bekaert)

# Drop missing values (weekends, holidays)
rf_fred = rf_fred.dropna().rename(columns={'DGS1MO': 'rf_annual_percent'})

# Convert from annual percent to daily decimal return
rf_fred['rf_daily'] = (1 + rf_fred['rf_annual_percent'] / 100) ** (1/252) - 1


# Créer le régime de volatilité selon le VIX (seuil p75)
seuil_p75 = vix['VIXCLS'].quantile(0.75)
vix['regime_vol'] = (vix['VIXCLS'] > seuil_p75).astype(int)

# Joindre toutes les séries sur l'index (date)
data = rf_fred.join(vix[['VIXCLS', 'regime_vol']], how='left')

# Préparer le DataFrame FRED
fred_df = data.reset_index()
fred_df = fred_df.rename(columns={'DATE': 'date'})
fred_df['date'] = pd.to_datetime(fred_df['date'])

# Fusionner sur la colonne date (jointure gauche pour garder toutes les lignes de df1)
dfr_fred = pd.merge(df, fred_df, on='date', how='left')

In [ ]:
del data

In [ ]:
del df

In [ ]:
start = datetime.datetime(2005, 1, 1)
end = datetime.datetime(2025, 1, 1)

dfr_fred = dfr_fred.dropna(subset=['date'])

taux_directeur = web.DataReader('FEDFUNDS', 'fred', start, end) #taux directeur fed

# Trie les deux tables par date
dfr_fred = dfr_fred.sort_values('date')

taux_directeur = taux_directeur.reset_index()

taux_directeur = taux_directeur.rename(columns={'DATE': 'date', 'FEDFUNDS': 'taux_direc'})
taux_directeur = taux_directeur.sort_values('date')

# Merge asof (prend la dernière valeur FEDFUNDS connue avant ou à la date CRSP)
dfr_fred = pd.merge_asof(
    dfr_fred,
    taux_directeur,
    on='date',
    direction='backward'  ) # prend le dernier taux FEDFUNDS <= date CRSP)

In [ ]:
# Calcul du rendement excédentaire
dfr_fred['ret_exc'] = dfr_fred['ret'] - dfr_fred['rf_daily']

crisis_periods = [
    # Subprimes / Global Financial Crisis
    ('2007-07-01', '2009-06-30'),
    # European Sovereign Debt Crisis
    ('2010-05-01', '2012-12-31'),
    # China stock market crash
    ('2015-06-01', '2015-09-30'),
    # COVID-19 market crash
    ('2020-02-15', '2020-05-31'),
    # Inflation, Ukraine war, banking stress
    ('2022-02-24', '2023-06-30'),
]

dfr_fred['date'] = pd.to_datetime(dfr_fred['date'])

# Création du masque global
mask = pd.Series(False, index=dfr_fred.index)
for start, end in crisis_periods:
    mask = mask | dfr_fred['date'].between(pd.to_datetime(start), pd.to_datetime(end))

dfr_fred['crise'] = mask.astype(int)

## Mesure d'impact et statistiques statistiques descriptives

In [ ]:
# spread impact
dfr_fred['Spread_bid_ask_Impact'] = (dfr_fred['Spread_bid_ask'] / dfr_fred['echange_val'])*10000000
dfr_fred['Spread_hi_lo_Impact'] = (dfr_fred['Spread_hi_lo'] / dfr_fred['echange_val'])*10000000
# roll measure Impact
dfr_fred['roll_measure_Impact'] = (dfr_fred['roll_measure'] / dfr_fred['echange_val'])*10000000
# Mesure Amihud
dfr_fred['Amihud'] = (np.abs(dfr_fred['ret']) / dfr_fred['echange_val'])*10000000
# Calcul mesure Corwin_Schultz
dfr_fred['Corwin_Schultz'] = (dfr_fred['S'] / dfr_fred['echange_val'])*10000000

In [ ]:
dfr_fred['lag_size'] = dfr_fred.groupby('RIC')['size'].shift(1)
dfr_fred['lag_capex'] = dfr_fred.groupby('RIC')['capex'].shift(1)
dfr_fred['lag_leverage'] = dfr_fred.groupby('RIC')['leverage'].shift(1)
dfr_fred['lag_liquidity'] = dfr_fred.groupby('RIC')['liquidity'].shift(1)
dfr_fred['lag_ROA'] = dfr_fred.groupby('RIC')['ROA'].shift(1)
dfr_fred['lag_price_to_book'] = dfr_fred.groupby('RIC')['price_to_book'].shift(1)
dfr_fred['lag_pe_ratio'] = dfr_fred.groupby('RIC')['pe_ratio'].shift(1)
dfr_fred['lag_ret'] = dfr_fred.groupby('RIC')['ret'].shift(1)


dfr_fred['lag_Spread_bid_ask'] = dfr_fred.groupby('RIC')['Spread_bid_ask'].shift(1)
dfr_fred['lag2_Spread_bid_ask'] = dfr_fred.groupby('RIC')['lag_Spread_bid_ask'].shift(1)
dfr_fred['lag3_Spread_bid_ask'] = dfr_fred.groupby('RIC')['lag2_Spread_bid_ask'].shift(1)
dfr_fred['lag4_Spread_bid_ask'] = dfr_fred.groupby('RIC')['lag3_Spread_bid_ask'].shift(1)
dfr_fred['lag5_Spread_bid_ask'] = dfr_fred.groupby('RIC')['lag4_Spread_bid_ask'].shift(1)
dfr_fred['lag6_Spread_bid_ask'] = dfr_fred.groupby('RIC')['lag5_Spread_bid_ask'].shift(1)

dfr_fred['lag_Spread_hi_lo'] = dfr_fred.groupby('RIC')['Spread_hi_lo'].shift(1)
dfr_fred['lag2_Spread_hi_lo'] = dfr_fred.groupby('RIC')['lag_Spread_hi_lo'].shift(1)
dfr_fred['lag3_Spread_hi_lo'] = dfr_fred.groupby('RIC')['lag2_Spread_hi_lo'].shift(1)
dfr_fred['lag4_Spread_hi_lo'] = dfr_fred.groupby('RIC')['lag3_Spread_hi_lo'].shift(1)
dfr_fred['lag5_Spread_hi_lo'] = dfr_fred.groupby('RIC')['lag4_Spread_hi_lo'].shift(1)
dfr_fred['lag6_Spread_hi_lo'] = dfr_fred.groupby('RIC')['lag5_Spread_hi_lo'].shift(1)

dfr_fred['lag_roll_measure'] = dfr_fred.groupby('RIC')['roll_measure'].shift(1)
dfr_fred['lag2_roll_measure'] = dfr_fred.groupby('RIC')['lag_roll_measure'].shift(1)
dfr_fred['lag3_roll_measure'] = dfr_fred.groupby('RIC')['lag2_roll_measure'].shift(1)
dfr_fred['lag4_roll_measure'] = dfr_fred.groupby('RIC')['lag3_roll_measure'].shift(1)
dfr_fred['lag5_roll_measure'] = dfr_fred.groupby('RIC')['lag4_roll_measure'].shift(1)
dfr_fred['lag6_roll_measure'] = dfr_fred.groupby('RIC')['lag5_roll_measure'].shift(1)

dfr_fred['lag_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['Spread_bid_ask_Impact'].shift(1)
dfr_fred['lag2_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['lag_Spread_bid_ask_Impact'].shift(1)
dfr_fred['lag3_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['lag2_Spread_bid_ask_Impact'].shift(1)
dfr_fred['lag4_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['lag3_Spread_bid_ask_Impact'].shift(1)
dfr_fred['lag5_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['lag4_Spread_bid_ask_Impact'].shift(1)
dfr_fred['lag6_Spread_bid_ask_Impact'] = dfr_fred.groupby('RIC')['lag5_Spread_bid_ask_Impact'].shift(1)

dfr_fred['lag_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['Spread_hi_lo_Impact'].shift(1)
dfr_fred['lag2_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['lag_Spread_hi_lo_Impact'].shift(1)
dfr_fred['lag3_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['lag2_Spread_hi_lo_Impact'].shift(1)
dfr_fred['lag4_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['lag3_Spread_hi_lo_Impact'].shift(1)
dfr_fred['lag5_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['lag4_Spread_hi_lo_Impact'].shift(1)
dfr_fred['lag6_Spread_hi_lo_Impact'] = dfr_fred.groupby('RIC')['lag5_Spread_hi_lo_Impact'].shift(1)

dfr_fred['lag_roll_measure_Impact'] = dfr_fred.groupby('RIC')['roll_measure_Impact'].shift(1)
dfr_fred['lag2_roll_measure_Impact'] = dfr_fred.groupby('RIC')['lag_roll_measure_Impact'].shift(1)
dfr_fred['lag3_roll_measure_Impact'] = dfr_fred.groupby('RIC')['lag2_roll_measure_Impact'].shift(1)
dfr_fred['lag4_roll_measure_Impact'] = dfr_fred.groupby('RIC')['lag3_roll_measure_Impact'].shift(1)
dfr_fred['lag5_roll_measure_Impact'] = dfr_fred.groupby('RIC')['lag4_roll_measure_Impact'].shift(1)
dfr_fred['lag6_roll_measure_Impact'] = dfr_fred.groupby('RIC')['lag5_roll_measure_Impact'].shift(1)

dfr_fred['lag_Amihud'] = dfr_fred.groupby('RIC')['Amihud'].shift(1)
dfr_fred['lag2_Amihud'] = dfr_fred.groupby('RIC')['lag_Amihud'].shift(1)
dfr_fred['lag3_Amihud'] = dfr_fred.groupby('RIC')['lag2_Amihud'].shift(1)
dfr_fred['lag4_Amihud'] = dfr_fred.groupby('RIC')['lag3_Amihud'].shift(1)
dfr_fred['lag5_Amihud'] = dfr_fred.groupby('RIC')['lag4_Amihud'].shift(1)
dfr_fred['lag6_Amihud'] = dfr_fred.groupby('RIC')['lag5_Amihud'].shift(1)

dfr_fred['lag_Corwin_Schultz'] = dfr_fred.groupby('RIC')['Corwin_Schultz'].shift(1)
dfr_fred['lag2_Corwin_Schultz'] = dfr_fred.groupby('RIC')['lag_Corwin_Schultz'].shift(1)
dfr_fred['lag3_Corwin_Schultz'] = dfr_fred.groupby('RIC')['lag2_Corwin_Schultz'].shift(1)
dfr_fred['lag4_Corwin_Schultz'] = dfr_fred.groupby('RIC')['lag3_Corwin_Schultz'].shift(1)
dfr_fred['lag5_Corwin_Schultz'] = dfr_fred.groupby('RIC')['lag4_Corwin_Schultz'].shift(1)
dfr_fred['lag6_Corwin_Schultz'] = dfr_fred.groupby('RIC')['lag5_Corwin_Schultz'].shift(1)

In [ ]:
# On Winsorise les variables de cout de transaction
cols = ["Spread_bid_ask", "Spread_hi_lo", "roll_measure",
        "Spread_bid_ask_Impact", "Spread_hi_lo_Impact",
        "roll_measure_Impact", "Amihud", "Corwin_Schultz"]

# Winsorisation à 1% et 99%
for col in cols:
    lower = dfr_fred[col].quantile(0.01)   # borne basse
    upper = dfr_fred[col].quantile(0.99)   # borne haute
    dfr_fred[col] = dfr_fred[col].clip(lower, upper)  # coupe les valeurs extrêmes

In [ ]:
dfr_fred['Amihud'] = dfr_fred['Amihud'].replace(0, np.nan)
dfr_fred['profond_market'] = np.log(1 / dfr_fred['Amihud'])

dfr_fred['price_to_book_log'] = np.nan  # initialise
dfr_fred.loc[dfr_fred['price_to_book'] > 0, 'price_to_book_log'] = np.log(
    dfr_fred.loc[dfr_fred['price_to_book'] > 0, 'price_to_book']
)

In [ ]:
start = datetime.datetime(2005, 1, 1)
end = datetime.datetime(2025, 1, 1)
# 2. Trier par RIC puis par date (ordre croissant par défaut)
dfr_fred.sort_values(by=['RIC', 'date'], inplace=True)

# 3. Supprimer les doublons (garde la première occurrence)
dfr_fred.drop_duplicates(subset=['RIC', 'date'], keep='first', inplace=True)

# Filtrer les dates entre 1er janvier 2005 et 1er janvier 2025
dfr_fred = dfr_fred[(dfr_fred['date'] >= start) & (dfr_fred['date'] <= end)]

In [ ]:
# exporter dfr_fred en dfr_large.parquet
chemin = r"C:/Users/Subtech/Desktop/data_final_memoire/dfr_large.parquet"

# Export en Parquet avec compression Snappy (rapide et compressé)
dfr_fred.to_parquet(chemin, engine="pyarrow", compression='snappy',  index=True)

In [ ]:
dfr_fred

# II. TESTER L'HYPOTHESE 1

In [3]:
import numpy as np
import pandas as pd
import datetime
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IVGMM
from linearmodels import PanelOLS
from linearmodels import IV2SLS
from collections import OrderedDict
from linearmodels.panel import compare
import pandas_datareader.data as web
from scipy.stats.mstats import winsorize

In [5]:
chemin = r"/content/dfr_large.parquet"

dfr_large = pd.read_parquet(chemin, engine='pyarrow')

FileNotFoundError: [Errno 2] No such file or directory: '/content/dfr_large.parquet'

In [ ]:
# Sélectionne les colonnes d'intérêt
cols = ['Spread_bid_ask', 'Spread_hi_lo', 'roll_measure', 'Spread_bid_ask_Impact', 'Spread_hi_lo_Impact', 'roll_measure_Impact',  'Amihud', 'Corwin_Schultz']

# Calcul de la matrice de corrélation
dfr_fred[cols].corr()

In [ ]:
dfr_large[['prc_close', 'prc_open', 'bid', 'ask', 'prc_high', 'prc_low', 'volume', 'turnover', 'echange_val', 'pe_ratio', 'price_to_book', 'tot_asset', 'tot_liab', 'net_income', 'capex', 'tot_cur_asset', 'tot_cur_liab', 'size', 'ROA', 'leverage', 'liquidity']].describe()

In [ ]:
dfr_fred[['Spread_bid_ask', 'Spread_hi_lo', 'roll_measure', 'Spread_bid_ask_Impact', 'Spread_hi_lo_Impact', 'roll_measure_Impact', 'Amihud', 'Corwin_Schultz']].describe()

### H1 : Les coûts de transaction ont un effet significatif sur les rendements des actions et leur volatilité.

In [ ]:
#dfr_fred[["roll_measure_Impact"] + instruments].corr()

In [ ]:
controls = ['size', 'price_to_book_log', 'leverage', 'lag_ROA']
dfr_large["const"] = 1
controls = ["const"] + controls

In [ ]:
instruments = ['lag_Spread_bid_ask', 'lag2_Spread_bid_ask', 'lag3_Spread_bid_ask', 'lag4_Spread_bid_ask', 'lag5_Spread_bid_ask',  'lag6_Spread_bid_ask']
# dropna avant estimation implace
dfgm1 = dfr_large.dropna(subset=['ret_exc', 'Spread_bid_ask'] + controls + instruments)

gmmod = IVGMM(
    dfgm1.ret_exc, # dependante
    dfgm1[controls], # controle
    dfgm1.Spread_bid_ask, # endogene
    dfgm1[instruments], # instruments
    weight_type="clustered",
    clusters=dfgm1.index.get_level_values("RIC")
    )
gmm1 = gmmod.fit()
gmm1

In [ ]:
gmm1.j_stat

In [ ]:
instruments = [ 'lag_Spread_hi_lo', 'lag2_Spread_hi_lo', 'lag3_Spread_hi_lo', 'lag4_Spread_hi_lo', 'lag5_Spread_hi_lo', 'lag6_Spread_hi_lo']
# dropna avant estimation implace
dfgm2 = dfr_large.dropna(subset=['ret_exc', 'Spread_hi_lo'] + controls + instruments)

gmmod = IVGMM(
    dfgm2.ret_exc,  # dependante
    dfgm2[controls], # controle
    dfgm2.Spread_hi_lo,  # endogene
    dfgm2[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm2.index.get_level_values("RIC")
    )
gmm2 = gmmod.fit()
gmm2

In [ ]:
gmm2.j_stat

In [ ]:
instruments = ['lag_roll_measure', 'lag2_roll_measure', 'lag3_roll_measure', 'lag4_roll_measure', 'lag5_roll_measure', 'lag6_roll_measure']
# dropna avant estimation implace
dfgm3 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)


gmmod = IVGMM(
    dfgm3.ret_exc,  # dependante
    dfgm3[controls], # controle
    dfgm3.roll_measure,  # endogene
    dfgm3[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm3.index.get_level_values("RIC")
    )
gmm3 = gmmod.fit()
gmm3

In [ ]:
gmm3.j_stat

In [ ]:
instruments = ['lag_Spread_bid_ask_Impact', 'lag2_Spread_bid_ask_Impact', 'lag3_Spread_bid_ask_Impact', 'lag4_Spread_bid_ask_Impact', 'lag5_Spread_bid_ask_Impact', 'lag6_Spread_bid_ask_Impact']
# dropna avant estimation implace
dfgm4 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)

gmmod = IVGMM(
    dfgm4.ret_exc,  # dependante
    dfgm4[controls], # controle
    dfgm4.Spread_bid_ask_Impact,  # endogene
    dfgm4[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm4.index.get_level_values("RIC")
    )
gmm4 = gmmod.fit()
gmm4

In [ ]:
gmm4.j_stat

In [ ]:
instruments = ['lag_Spread_hi_lo_Impact', 'lag2_Spread_hi_lo_Impact', 'lag3_Spread_hi_lo_Impact', 'lag4_Spread_hi_lo_Impact', 'lag5_Spread_hi_lo_Impact', 'lag6_Spread_hi_lo_Impact']
# dropna avant estimation implace
dfgm5 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)

gmmod = IVGMM(
    dfgm5.ret_exc,  # dependante
    dfgm5[controls], # controle
    dfgm5.Spread_hi_lo_Impact,  # endogene
    dfgm5[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm5.index.get_level_values("RIC")
    )
gmm5 = gmmod.fit()
gmm5

In [ ]:
gmm5.j_stat

In [ ]:
instruments = ['lag_roll_measure_Impact', 'lag2_roll_measure_Impact', 'lag3_roll_measure_Impact', 'lag4_roll_measure_Impact', 'lag5_roll_measure_Impact', 'lag6_roll_measure_Impact']
# dropna avant estimation implace
dfgm6 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)

gmmod = IVGMM(
    dfgm6.ret_exc,  # dependante
    dfgm6[controls], # controle
    dfgm6.roll_measure_Impact,  # endogene
    dfgm6[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm6.index.get_level_values("RIC")
    )
gmm6 = gmmod.fit()
gmm6

In [ ]:
gmm6.j_stat

In [ ]:
instruments = [ 'lag_Amihud', 'lag2_Amihud', 'lag3_Amihud', 'lag4_Amihud', 'lag5_Amihud', 'lag6_Amihud']
# dropna avant estimation implace
dfgm7 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)

gmmod = IVGMM(
    dfgm7.ret_exc,  # dependante
    dfgm7[controls], # controle
    dfgm7.Amihud,  # endogene
    dfgm7[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm7.index.get_level_values("RIC")
    )
gmm7 = gmmod.fit()
gmm7

In [ ]:
gmm7.j_stat

In [ ]:
instruments = [ 'lag_Corwin_Schultz', 'lag2_Corwin_Schultz', 'lag3_Corwin_Schultz', 'lag4_Corwin_Schultz', 'lag5_Corwin_Schultz', 'lag6_Corwin_Schultz']
# dropna avant estimation implace
dfgm8 = dfr_large.dropna(subset=['ret_exc', 'roll_measure'] + controls + instruments)

gmmod = IVGMM(
    dfgm8.ret_exc,  # dependante
    dfgm8[controls], # controle
    dfgm8.Corwin_Schultz,  # endogene
    dfgm8[instruments], #instruments
    weight_type="clustered",
    clusters=dfgm8.index.get_level_values("RIC")
    )
gmm8 = gmmod.fit()
gmm8

In [ ]:
gmm8.j_stat

In [ ]:
from collections import OrderedDict
from linearmodels.iv.results import compare

res = OrderedDict()
res["gmm1"] = gmm1
res["gmm2"] = gmm2
res["gmm3"] = gmm3
res["gmm4"] = gmm4
res["gmm5"] = gmm5
res["gmm6"] = gmm6
res["gmm7"] = gmm7
res["gmm8"] = gmm8
print(compare(res))

In [ ]:

# Firm only
mod1 = PanelOLS.from_formula("ret_exc ~ Spread_bid_ask +  size + price_to_book + leverage + pe_ratio + lag_ROA + lag_capex + EntityEffects", dfr_fred)
res1 = mod1.fit(
    cov_type="clustered", cluster_entity=True, cluster_time=False, group_debias=True)


res1.summary
## GMM

# III. TESTER L'HYPOTHESE 2

### H2 : Cet effet est plus marqué dans les marchés émergents que dans les marchés développés, en raison des différences de liquidité, de profondeur de marché et de sophistication des investisseurs.

# IV. TESTER L'HYPOTHESE 3

### H3 : La sensibilité des rendements aux coûts de transaction varie selon le régime de marché (haute vs. basse volatilité) et les conditions macroéconomiques.